# Feature engineering — Base 5 — ouro semanal

Chama `preparar_ouro_semanal` e `construir_quadro_ouro`. As 49 features são as mesmas do Random Forest.

## Disponibilidade

- Preço e cobertura: aprovados na origem semanal (`W-FRI`), com `ffill` só do passado.
- `TREASURY_10Y` e `FED_FUNDS_RATE`: **provisórios** até confirmar fonte/atraso de publicação.
- `TARGET` e derivações diárias do CSV: **proibidas** em `X`.
- Arquivos: `catalogo_features_base5.csv` e `disponibilidade_covariaveis_base5.csv`.


In [1]:
from pathlib import Path
import sys

import pandas as pd

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "projeto.yaml").is_file():
            return candidate
    raise FileNotFoundError("Execute a partir do projeto ou de uma subpasta dele.")

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "src"))
from series_temporais.data.preparacao_base5 import construir_quadro_ouro, preparar_ouro_semanal
from series_temporais.features.regras_bases import catalogo_base5

BASE_DIR = ROOT / "data" / "base_05"
DATA_PATH = BASE_DIR / "raw.csv"
CATALOGO_PATH = BASE_DIR / "catalogo_features_base5.csv"
DISPONIBILIDADE_PATH = BASE_DIR / "disponibilidade_covariaveis_base5.csv"
TRAIN_RATIO = 0.75


## Série semanal e quadro

Semanas sem cotação carregam somente o último estado já conhecido. O retorno, os lags e as janelas são calculados depois dessa agregação.

In [2]:
bruto = pd.read_csv(DATA_PATH)
derivadas_entregues = set(bruto.columns[4:])
semanal = preparar_ouro_semanal(bruto)
quadro, features = construir_quadro_ouro(semanal)
assert len(semanal) == 2398
assert len(quadro) == 2344
assert len(features) == 49
assert derivadas_entregues.isdisjoint(features)
assert {
    "target_date",
    "target_price_t_plus_1",
    "target_log_return_t_plus_1",
    "target_has_new_quote",
}.isdisjoint(features)
quadro[features].head()


,price_t,treasury_10y_t,fed_funds_rate_t,log_price_t,log_return_t,source_observations,has_source_observation_t,price_was_carried,quote_age_days,treasury_change_t,...,treasury_lag_1,treasury_lag_4,treasury_lag_13,fed_funds_lag_1,fed_funds_lag_4,fed_funds_lag_13,target_week_sin,target_week_cos,target_month_sin,target_month_cos
0,43.60,6.20,8.38,3.775057,0.000000,4,1,0,0,0.00,...,6.20,6.25,6.20,8.00,6.63,6.75,0.663123,-0.748511,0.866025,-0.500000
1,43.50,6.26,8.50,3.772761,-0.002296,5,1,0,0,0.06,...,6.20,6.13,6.09,8.38,7.75,6.75,0.568065,-0.822984,0.866025,-0.500000
2,43.45,6.36,8.50,3.771611,-0.001150,4,1,0,1,0.10,...,6.26,6.14,6.23,8.50,7.75,6.75,0.464723,-0.885456,0.866025,-0.500000
3,43.45,6.36,8.50,3.771611,0.000000,0,0,1,8,0.00,...,6.36,6.20,6.27,8.50,8.00,6.63,0.354605,-0.935016,0.500000,-0.866025
4,40.75,6.54,9.63,3.707456,-0.064155,4,1,0,0,0.18,...,6.36,6.20,6.26,8.50,8.38,6.88,0.239316,-0.970942,0.500000,-0.866025


## Catálogo

In [3]:
catalogo = catalogo_base5(features)
assert not catalogo.grupo.eq("revisar").any()
assert catalogo.set_index("feature").loc["treasury_10y_t", "status"] == "provisorio"
disponibilidade = pd.DataFrame(
    [
        {
            "coluna": "TREASURY_10Y",
            "disponibilidade": "na_origem",
            "atraso": 0,
            "feature": "treasury_10y_t",
            "status": "provisorio",
            "justificativa": "Fonte/unidade/atraso de publicação pendentes.",
        },
        {
            "coluna": "FED_FUNDS_RATE",
            "disponibilidade": "na_origem",
            "atraso": 0,
            "feature": "fed_funds_rate_t",
            "status": "provisorio",
            "justificativa": "Fonte/unidade/atraso de publicação pendentes.",
        },
        {
            "coluna": "TARGET",
            "disponibilidade": "proibida",
            "atraso": 0,
            "feature": "",
            "status": "excluido",
            "justificativa": "Alvo diário inconsistente; fora de X.",
        },
        {
            "coluna": "GOLD_LAG_1",
            "disponibilidade": "proibida",
            "atraso": 0,
            "feature": "",
            "status": "excluido",
            "justificativa": "Derivação diária entregue; reconstruída semanalmente.",
        },
    ]
)
catalogo.to_csv(CATALOGO_PATH, index=False)
disponibilidade.to_csv(DISPONIBILIDADE_PATH, index=False)
catalogo.loc[catalogo.status.eq("provisorio")]


,feature,grupo,disponibilidade,descricao,status,nota
1,treasury_10y_t,estado_na_origem,conhecido em t,Valor observado na origem para prever o passo ...,provisorio,Fonte/unidade/atraso de publicação ainda pende...
2,fed_funds_rate_t,estado_na_origem,conhecido em t,Valor observado na origem para prever o passo ...,provisorio,Fonte/unidade/atraso de publicação ainda pende...
9,treasury_change_t,variacao_na_origem,conhecida em t,Variação calculada até a origem.,provisorio,Derivada da taxa provisória na origem.
10,fed_funds_change_t,variacao_na_origem,conhecida em t,Variação calculada até a origem.,provisorio,Derivada da taxa provisória na origem.


## Corte cronológico

O corte de 75% remove a origem cujo alvo cai na primeira semana de teste.

In [4]:
corte = int(len(quadro) * TRAIN_RATIO)
inicio_teste = quadro.iloc[corte].DATE
treino = quadro.iloc[:corte].loc[lambda x: x.target_date < inicio_teste].reset_index(drop=True)
teste = quadro.iloc[corte:].reset_index(drop=True)
assert len(treino) == 1757
assert len(teste) == 586
assert treino.target_date.max() < teste.DATE.min()
pd.DataFrame({
    "recorte": ["treino", "teste"],
    "linhas": [len(treino), len(teste)],
    "primeira_origem": [treino.DATE.min(), teste.DATE.min()],
    "ultima_origem": [treino.DATE.max(), teste.DATE.max()],
})


,recorte,linhas,primeira_origem,ultima_origem
0,treino,1757,1969-05-09,2003-01-03
1,teste,586,2003-01-17,2014-04-04


## Checagem de leakage

O preço e as taxas são alterados só depois de uma sexta-feira. As features das semanas anteriores não podem mudar.

In [5]:
indice = 1800
limite = semanal.loc[indice - 1, "DATE"]
alterado = semanal.copy()
alterado.loc[indice:, ["price_t", "log_price_t", "log_return_t", "treasury_10y_t", "fed_funds_rate_t"]] = 1_000_000_000
reconstruido, _ = construir_quadro_ouro(alterado)
antes = quadro.loc[quadro.DATE.le(limite), features].reset_index(drop=True)
depois = reconstruido.loc[reconstruido.DATE.le(limite), features].reset_index(drop=True)
assert antes.equals(depois)
{"origens_comparadas": len(antes), "diferenca_maxima": 0.0, "iguais": True}


{'origens_comparadas': 1747, 'diferenca_maxima': 0.0, 'iguais': True}

## Onde ficou

- Preparação semanal e as 49 features: `src/series_temporais/data/preparacao_base5.py`
- Catálogo: `data/base_05/catalogo_features_base5.csv`
- O notebook `grupo5_RF.ipynb` continua responsável pelo modelo e chama as mesmas funções.
